# Building a Python Fixing AI Agent using Lang Frameworks — Google Colab

This notebook does **one thing**: it runs the three finished agents against a real model, in the
browser.

It exists for the `colab` tier — machines that cannot comfortably hold a local model. You still do
the course the normal way, in the IDE, on your own machine:

- read every lesson,
- write the parts no framework writes for you (lesson 3: the graph's routing and the loop guard;
  lesson 4: what counts as acting, the idle counter, the nudge, the routing tail),
- run the exercise tests locally — they use a scripted **fake** model, so they need no model at all
  and they pass on any machine.

The only step that needs a real model is the last one in each lesson: watching the agent actually
fix a bug. That step happens here.

| Lesson | Repository | CLI | Framework | Reasoning |
|---|---|---|---|---|
| 2 — Agent with no framework | `agentfix-workshop` | `agentfix` | none — a `for` loop | no |
| 3 — What about frameworks? | `agentfix-langchain` | `agentfix` | LangGraph + LangChain | no |
| 4 — What about thinking? | `agentfix-react` | `agentgraph` | LangGraph + LangChain | **yes** |

**Do not run `python run.py doctor` on your own machine.** It will fail, because there is no Ollama
and no model there — expected and fine on this tier. The doctor checks that matter for you run in
this notebook, one per edition.

**The code you run here is the reference solution, not your own edits.** Section 4 checks out the
finished version of each exercise file so every agent below is guaranteed to run. Your own
implementation stays on your machine, verified by the exercise tests you already ran there.

## 0. Before you run anything

In Colab choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order, top to
bottom. CPU also works; it is just slower.

Nothing here needs editing — there are no exercises in this notebook. Budget roughly 30 minutes,
most of it the two model pulls and the live runs.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU — CPU inference will work, just slower."

## 1. Configuration

One model for all three lessons:

| Lessons | Model | Why |
|---|---|---|
| 2, 3 and 4 | `qwen3:1.7b`, derived as `agentfix-qwen3` | the smallest model that both **thinks** and calls tools |

Lesson 4 needs a model that reasons. A model with no thinking mode still completes every run —
silently, as the Act-only agent from lesson 3 — and "this agent does not reason" becomes a fact
about your setup rather than about the model. `agentgraph doctor` fails rather than letting that
pass quietly. `qwen3:1.7b` thinks *and* calls tools, so it covers the earlier lessons too and this
notebook needs a single pull.

On a local machine `./setup.sh` installs the Mellum2 pair (8 GB each) and records them in two
variables: `MELLUM_MODEL` for lessons 2–3, `AGENTGRAPH_MODEL` for lesson 4, because on that tier
one variable cannot name both a coding model and a thinking one. Here the same split is done per cell, and the
qwen pair above stands in for both — about 2.5 GB instead of 16.

In [ ]:
%env OLLAMA_CONTEXT_LENGTH=16384

!echo "workshop   https://github.com/jelenadjuric01/agentfix-workshop.git   -> /content/agentfix-workshop"
!echo "langchain  https://github.com/jelenadjuric01/agentfix-langchain.git  -> /content/agentfix-langchain"
!echo "react      https://github.com/jelenadjuric01/agentfix-react.git      -> /content/agentfix-react"
!echo
!echo "lessons 2-4     qwen3:1.7b -> agentfix-qwen3"
!echo "context length  16384"

## 2. Install and start Ollama

Colab may not include `zstd`, which the current Ollama Linux installer needs — without it the
install stops with an error about zstd that reads like a broken download.

The server is started in the background and stays available to every later cell.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd pciutils
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; fi
!ollama --version

In [ ]:
!if ! curl -fsS http://127.0.0.1:11434/api/version >/dev/null 2>&1; then OLLAMA_MAX_LOADED_MODELS=1 nohup ollama serve > /tmp/ollama-colab.log 2>&1 & sleep 4; fi
!curl -fsS http://127.0.0.1:11434/api/version

## 3. Pull the model

Do not skip the `ollama create` steps. They bake `num_ctx 16384` into a derived model with a stable
name — too small a context window does not error, it silently truncates the agent's history, which
looks like a stupid model rather than a misconfiguration.

Lesson 4's client talks to Ollama's **native** API, which honours a per-request `num_ctx`, so
`agentfix-qwen3` would work without deriving it. It is derived anyway, so the notebook and
`setup.py` produce the same model name on every machine.

The server above was started with `OLLAMA_MAX_LOADED_MODELS=1`: one model in memory at a time. On a
16 GB laptop with the Mellum2 pair that setting is the difference between working and swapping, and
it costs nothing here.

This cell downloads about 1.4 GB, so it is the slow one.

In [ ]:
!ollama pull qwen3:1.7b
!printf "FROM qwen3:1.7b\nPARAMETER num_ctx 16384\n" > /tmp/Modelfile.agentfix-qwen3
!ollama create agentfix-qwen3 -f /tmp/Modelfile.agentfix-qwen3

!ollama list

In [ ]:
# Smoke test: expect a <think> block, then READY.
!ollama run agentfix-qwen3 "Reply with exactly: READY"

## 4. Clone the three repositories, finished

Each edition gets its own clone under `/content`, from `main` — the exercise branch — and then the
one file each lesson stubs out is checked out from that lesson's solution tag:

| Repository | File | From |
|---|---|---|
| `agentfix-workshop` | `tools/tests_tool.py`, `agent/loop.py` | `stage-3-solution` |
| `agentfix-langchain` | `agent/graph.py` | `stage-2-solution` |
| `agentfix-react` | `agent/graph.py` | `stage-1-solution` |

This is the opposite of what the IDE course does, and deliberately so: **this notebook runs the
agents, it does not grade you.** Nothing here is checked, nothing here is your work, and a solution
in this runtime cannot overwrite anything on your machine. Push is disabled on every clone.

In [ ]:
%cd /content

!for r in agentfix-workshop agentfix-langchain agentfix-react; do \
    rm -rf /content/$r; \
    git clone -q --branch main https://github.com/jelenadjuric01/$r.git /content/$r; \
    git -C /content/$r fetch -q --all --tags --prune; \
    git -C /content/$r remote set-url --push origin DISABLED; \
  done

!git -C /content/agentfix-workshop  checkout -q stage-3-solution -- src/agentfix/tools/tests_tool.py src/agentfix/agent/loop.py
!git -C /content/agentfix-langchain checkout -q stage-2-solution -- src/agentfix/agent/graph.py
!git -C /content/agentfix-react     checkout -q stage-1-solution -- src/agentgraph/agent/graph.py

!for r in agentfix-workshop agentfix-langchain agentfix-react; do \
    echo "$r  branch=$(git -C /content/$r branch --show-current)  HEAD=$(git -C /content/$r rev-parse --short HEAD)  push=$(git -C /content/$r remote get-url --push origin)"; \
  done

In [ ]:
# Guardrail: no stub marker may survive, or the agents below cannot run.
!if grep -rn "EXERCISE(stage-\|TODO(stage-\|NotImplementedError(\"stage" \
     /content/agentfix-workshop/src/agentfix/agent/loop.py \
     /content/agentfix-workshop/src/agentfix/tools/tests_tool.py \
     /content/agentfix-langchain/src/agentfix/agent/graph.py \
     /content/agentfix-react/src/agentgraph/agent/graph.py; then \
   echo "STOP: a stub survived — rerun the clone cell above"; \
 else \
   echo "OK: all three editions are complete."; \
 fi


---

# Lesson 2 — Agent with no framework

An agent is a while-loop around a chat model that can call functions and sees the result. Nothing
more magical than that. The loop in `src/agentfix/agent/loop.py` is about 15 lines; the rest of
`run_agent` is tracing and token accounting.

> **Note on the install.** This edition and the LangGraph edition are both packaged as `agentfix`,
> so installing one replaces the other. That is fine going through the notebook in order — each
> lesson reinstalls its own. Come back and rerun this cell if you want the no-framework agent again.

In [ ]:
%cd /content/agentfix-workshop
%env MELLUM_MODEL=agentfix-qwen3

!python -m pip uninstall -q -y agentfix
!python -m pip install -q -e .
!agentfix doctor

`doctor` should report `[PASS]` on everything except `ram` — Colab has less than 16 GB, which is
exactly why this notebook uses the small models. A `ram` FAIL here is expected and harmless. The
line to check is **`context window: 16384`**; if it says `4096`, rerun the `ollama create` cell in
section 3.

### Run it for real

`--verbose` prints the trace. You should see the model call `run_tests`, look around with
`list_files` / `read_file`, write a file, and run the tests again — that last call is what ends the
run, because the stop condition believes the test suite rather than the model.

A 1.5B model does not fix every task. `NOT SOLVED` after ten steps is not a broken setup; it is a
small model, and watching it fail is informative. The second task is the harder one: the bug is
**not** in the file the failing test points at, which is why `list_files` and `read_file` earn their
place.

In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose

In [ ]:
!agentfix solve tasks/workshop/02-invoice --verbose

---

# Lesson 3 — What about frameworks?

The same agent rebuilt on LangGraph for the graph and LangChain for the model and tool interfaces.
The question the lesson exists to answer: *which parts of my agent does a framework actually write
for me?*

**What it gives you:** `ToolNode` runs the calls (dispatch, ordering, unknown tool names, argument
validation, error recovery). `add_messages` makes the history append-only by construction, which
keeps the prompt prefix byte-stable and the server's KV cache valid. Reducers on `AgentState`
accumulate the counters. Callbacks carry the trace. The checkpointer snapshots state after every
node.

**What it does not:** the stop condition, the loop guard, and the step budget — `recursion_limit`
counts node executions, not model turns. Those are the two stages you write in the IDE.

In [ ]:
%cd /content/agentfix-langchain
%env MELLUM_MODEL=agentfix-qwen3

!python -m pip uninstall -q -y agentfix
!python -m pip install -q -e .
!agentfix doctor

In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose

Same agent, same tools, same model — the trace should look familiar, and that is the point. What
changed is who owns the plumbing.

The next cell runs the whole three-task workshop suite. It calls the model repeatedly, so it is
slow; skip it if you would rather get to lesson 4.

In [ ]:
!agentfix eval --suite workshop --limit 3

---

# Lesson 4 — What about thinking?

Same graph, driven by a model that **reasons before it acts**.

One flag on the client — `reasoning=True` — asks Ollama for the thinking and hands it back on **its
own channel** instead of leaving `<think>` tags inline in the answer. That channel matters more than
it sounds: with the tags inline, the model's deliberation ends up in the next prompt, in the trace,
and — the expensive one — inside the "complete file contents" that `write_file` is handed.

**There is no think step and no new node.** This is what ReAct means: the model thinks and acts in
the *same* turn. The graph from lesson 3 is unchanged in shape. What changed is two decisions about
what a turn *was*:

- **A turn with no tool call is no longer rare.** The Instruct model acted on every turn but the
  last. A thinking model will spend an entire turn reasoning and ask for nothing — and nudging that
  forever is an unbounded loop wearing a step budget as a disguise. Hence `idle_turns` and
  `MAX_IDLE_TURNS = 2`: a loop guard for thinking, alongside the one for actions.
- **The action guard must ignore reasoning.** `call_signature` still hashes only the tool name and
  arguments. A model that reasons its way to the same useless call by a fresh route every time is
  still stuck, and novel thinking must not buy a repeated call another turn.

Note the CLI name changes to `agentgraph`, and the model changes to the thinking one. Both variables
are set below: the published repo reads `MELLUM_MODEL`, and the IDE course's `setup.py` writes
`AGENTGRAPH_MODEL` — pointing them at the same model means either version behaves identically.

In [ ]:
%cd /content/agentfix-react
%env MELLUM_MODEL=agentfix-qwen3
%env AGENTGRAPH_MODEL=agentfix-qwen3

!python -m pip install -q -e .
!agentgraph doctor

Two of those checks are new, and they are the ones worth having, because both failures leave you
with a *working* agent that nothing else would complain about:

- **`reasoning`** — the model thinks, and the thinking arrives on its own channel. Fails distinctly
  if it is coming back inline as `<think>` tags.
- **`tool calling`** — it can still act while thinking. A model that reasons and calls nothing
  changes no files.

`ram` will FAIL on Colab as before; that is expected.

### Run the thinking agent for real

Every model turn now prints a `thinks` line above what it did. Read one — that text is the plan the
earlier editions never had. Two more things to watch for:

- `(NO REASONING)` now means what it says. In lesson 3 it appeared on almost every turn, because
  reasoning was read off `content`; here it prints only when the model genuinely skipped thinking.
- Reason twice with no tool call and the run ends with
  `abandoned — 2 consecutive turns with no tool call`. That is the Stage 1 guard, and it is the only
  line in the trace no tool produced.

Expect this to be **slower and more expensive per task** than lesson 3. That is the trade, and the
numbers below are where you see it.

In [ ]:
!agentgraph solve tasks/workshop/01-shopcart --verbose

In [ ]:
# Slow. Reasoning is generated tokens, and prior thoughts are re-sent on every later turn.
!agentgraph eval --suite workshop --limit 2

---

# What the numbers say

Your Colab runs use 1.5B and 1.7B stand-ins, so do not read your own pass rates as the course's
result. These are the shipped measurements from the reference machine (Apple M4, 24 GB, Mellum2
12B), on 20 HumanEvalFix tasks with the same 10-step budget — each repo ships them under
`results/precomputed/`:

| Edition | pass@1 | median steps | tokens | wall clock | peak prompt |
|---|---|---|---|---|---|
| No framework, Instruct | 0.60 (12/20) | 7 | 185,235 | 8m08s | 2,998 |
| LangGraph, Instruct | 0.45 (9/20) | 10 | 237,651 | 8m15s | 3,929 |
| **LangGraph, Thinking** | **0.80 (16/20)** | **5** | **415,333** | **52m25s** | **12,599** |

**Thinking is the largest single move in the course.** It did not just solve more, it solved in
*fewer* turns — fourteen of the sixteen successes took exactly five steps. And it is expensive:
1.75× the tokens for 6× the wall clock, and a peak prompt of 12,599 against a 16,384-token window —
three-quarters of the way to overflow on a benchmark of *small* bugs.

**Do not read 0.60 → 0.45 as a cost of the framework.** Temperature is 0.6 in all three, so a single
20-task run is noisy, and the two Instruct editions take identical step counts on the tasks they
both solve. In the no-framework edition, making the stop condition real moved pass@1 from 0.50 to
0.60 on its own — larger than the gap between those rows. What moves the number is the prompt, the
budget and the stop condition, not the plumbing.

In [ ]:
# Every repo ships its reference runs under results/precomputed/ (a live `eval` writes to
# results/, which is gitignored). Read whichever files the clone actually has.
# Braces are avoided on purpose: IPython substitutes them inside a ! command.
!for r in agentfix-workshop agentfix-langchain agentfix-react; do \
    echo "===== $r"; \
    for f in /content/$r/results/precomputed/*.json; do \
      python3 -c "import json,sys;d=json.load(open(sys.argv[1]));rs=d['results'];t=sum(x['prompt_tokens']+x['completion_tokens'] for x in rs);s=sum(x['duration_s'] for x in rs);print('  %-13s pass@1 %.2f   steps %-12s %7d tok  %2dm%02ds  peak %6d' % (d['suite'],d['pass_at_1'],[x['steps_used'] for x in rs],t,s//60,s%60,d['peak_prompt_tokens']))" "$f"; \
    done; \
  done

# Safety, briefly

The agent executes model-written code. Two boundaries, at two different layers, and neither changed
when the agent moved onto a framework or gained reasoning — confinement is a property of the tools
and the sandbox, not of the loop that calls them:

- **The tool layer confines paths.** `resolve_in_root` rejects any path that would escape the task's
  working directory *before* a read or write happens, and the write tool is constructed with the set
  of files that existed in the pristine template, so the agent cannot create a file and then start
  writing to it.
- **The sandbox confines execution.** The default backend is a hardened subprocess — stripped
  environment, resource limits, a timeout — which is **not** a security boundary. The Docker backend
  is: no network, memory/pid/CPU caps, non-root user, read-only mount.

Docker is not available in Colab, so this notebook runs the subprocess backend throughout. On your
own machine: `AGENTFIX_SANDBOX=docker` for lessons 2–3, `AGENTGRAPH_SANDBOX=docker` for lesson 4 —
each after building that edition's image, because the image name changed with the package.

In [ ]:
!echo "===== path confinement ====="; sed -n '/^def resolve_in_root/,/^def /p' /content/agentfix-react/src/agentgraph/tools/fs.py | head -30

# Where to go from here

Roughly in order of what would pay off next on the numbers above:

- **Context management** — trimming or summarising old turns, or dropping stale reasoning from the
  history. The clearest gap, and what stands between this agent and a task bigger than a one-file
  bug. Peak prompt was already at three-quarters of the window.
- **Planning as its own phase** — the model plans inside a turn now; nothing makes it commit to a
  plan across turns or notice when it has abandoned one.
- **Reflection / self-critique** — no separate pass where the model reviews its own diff before the
  tests do. Worth adding where tests are weak; worth being suspicious of where they are strong.
- **Parallel tool calls** — one at a time here, on purpose: `max_concurrency=1` is what keeps the
  test result honest. Doing it properly means knowing which calls are safe to overlap.
- **Multi-agent coordination** — one model, one graph, no delegation. The reason to want it is
  context; the reason to be careful is that every hand-off is a place where "done" can be claimed
  rather than verified.

If you take one thing from the whole course, make it the stop condition: three editions in, the
thing that decides whether an agent is trustworthy is still that it believes the test suite rather
than the model.

**Nothing to clean up here** — this runtime is discarded when the session ends. Delete the notebook
copy from your Drive if Colab saved one, and you are done. The IDE course's last lesson covers
taking the models off a real machine, and its `TROUBLESHOOT.md` covers what breaks.

The three repositories, if you would rather read them in your own IDE:

- https://github.com/jelenadjuric01/agentfix-workshop
- https://github.com/jelenadjuric01/agentfix-langchain
- https://github.com/jelenadjuric01/agentfix-react